In [2]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_parquet
from pyspark.sql.functions import col, dayofweek, hour, when, round, to_date, count, sum, avg, rank
from pyspark.sql import Window

# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [3]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\04_join_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\04_join_data\\taxi02")

In [5]:
taxi01_df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+------------------+----------+------------+-------------+----------+--------------------+---------------+-------------+----------+--------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_duration_minutes|average_speed_kmph|is_weekend|is_rush_hour|PU_LocationID|PU_borough|             PU_zone|PU_service_zone|DO_LocationID|DO_borough|             DO_zone|DO_service_zone|
+--------+--------------------+---------------------+---------------+-----------

In [6]:
# Top 10 drivers by revenue using window function
revenue_window = Window.orderBy(col("total_revenue").desc())

taxi01_top_drivers = taxi01_df.groupBy("VendorID").agg(
    sum("total_amount").alias("total_revenue")
).withColumn("rank", rank().over(revenue_window)).filter(col("rank") <= 10)

taxi02_top_drivers = taxi02_df.groupBy("VendorID").agg(
    sum("total_amount").alias("total_revenue")
).withColumn("rank", rank().over(revenue_window)).filter(col("rank") <= 10)


In [68]:
taxi02_top_drivers.show()


+--------+--------------------+----+
|VendorID|       total_revenue|rank|
+--------+--------------------+----+
|       2|   7.8988580290151E7|   1|
|       1|6.4397085720622174E7|   2|
+--------+--------------------+----+



In [7]:
# Rank zones by number of trips using window function
zone_window = Window.orderBy(col("total_trips").desc())

taxi01_ranked_zones = taxi01_df.groupBy("PULocationID", col("PU_zone").alias("zone_name")).agg(
    count("*").alias("total_trips")
).withColumn("rank", rank().over(zone_window)).orderBy("rank")

taxi02_ranked_zones = taxi02_df.groupBy("PULocationID", col("PU_zone").alias("zone_name")).agg(
    count("*").alias("total_trips")
).withColumn("rank", rank().over(zone_window)).orderBy("rank")


In [70]:
taxi01_ranked_zones.show(10)

+------------+--------------------+-----------+----+
|PULocationID|           zone_name|total_trips|rank|
+------------+--------------------+-----------+----+
|         237|Upper East Side S...|     380536|   1|
|         236|Upper East Side N...|     363565|   2|
|         161|      Midtown Center|     360911|   3|
|         186|Penn Station/Madi...|     339872|   4|
|         230|Times Sq/Theatre ...|     333954|   5|
|         234|            Union Sq|     332923|   6|
|         162|        Midtown East|     330847|   7|
|         170|         Murray Hill|     306133|   8|
|          79|        East Village|     303452|   9|
|          48|        Clinton East|     297312|  10|
+------------+--------------------+-----------+----+
only showing top 10 rows


In [8]:
# Running total of daily revenue using window function
running_window = Window.orderBy("trip_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

taxi01_running_revenue = taxi01_df.groupBy(
    to_date("tpep_pickup_datetime").alias("trip_date")
).agg(
    round(sum("total_amount"), 2).alias("daily_revenue")
).withColumn("running_total", round(sum("daily_revenue").over(running_window), 2)).orderBy("trip_date")

taxi02_running_revenue = taxi02_df.groupBy(
    to_date("tpep_pickup_datetime").alias("trip_date")
).agg(
    round(sum("total_amount"), 2).alias("daily_revenue")
).withColumn("running_total", round(sum("daily_revenue").over(running_window), 2)).orderBy("trip_date")


In [35]:
taxi01_running_revenue.show(10)

+----------+-------------+-------------+
| trip_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2017-01-01|   5012258.64|   5012258.64|
|2017-01-02|   3531850.21|   8544108.85|
|2017-01-03|   4270844.41|1.281495326E7|
|2017-01-04|    4451220.3|1.726617356E7|
|2017-01-05|   4827562.68|2.209373624E7|
|2017-01-06|   4964731.35|2.705846759E7|
|2017-01-07|   4331806.74|3.139027433E7|
|2017-01-08|   4528445.18|3.591871951E7|
|2017-01-09|   4793830.87|4.071255038E7|
|2017-01-10|    4742261.4|4.545481178E7|
+----------+-------------+-------------+
only showing top 10 rows


In [10]:
# Save Aggregated data
write_parquet(taxi01_running_revenue, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\06_window_data\\running_revenue")

write_parquet(taxi01_ranked_zones, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\06_window_data\\ranked_zones")

write_parquet(taxi01_top_drivers, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\06_window_data\\top_drivers")

print("Window function data saved successfully!")

Window function data saved successfully!


In [11]:
spark.stop()